# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, baselines, MOO, figures, download.

**Run cells in order.** After Cell 2, the repo is checked out. After Cell 5, weights are ready.

In [1]:
!pwd

/content


In [2]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 87.2 MB/s eta 0:00:00:00:0100:01
Cloning into '2601_chip_paper'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 101 (delta 41), reused 85 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 2.69 MiB | 7.63 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/2601_chip_paper


In [11]:
!git restore .

In [3]:
# Cell 2: Pull latest code and install all dependencies
!git pull
!uv sync  # Installs ALL deps from pyproject.toml including xgboost

Already up to date.
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 80 packages in 1ms
Prepared 75 packages in 12.09s                                           
Installed 75 packages in 466ms                              
 + about-time==4.2.1
 + alive-progress==3.3.0
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + autograd==1.8.0
 + cffi==2.0.0
 + click==8.3.3
 + cma==4.4.4
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + deprecated==1.3.1
 + executing==2.2.1
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.2.0
 + graphemeu==0.7.2
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.5.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mdurl==0.1.2
 + moocore==0.3.1
 + mpmath==1.3.0
 + nest-asyncio=

In [4]:
!uv remove torch

Resolved 69 packages in 905ms                                        
Uninstalled 10 packages in 247ms
 - filelock==3.25.2
 - fsspec==2026.2.0
 - jinja2==3.1.6
 - markupsafe==3.0.3
 - mpmath==1.3.0
 - networkx==3.6.1
 - setuptools==70.2.0
 - sympy==1.14.0
 - torch==2.11.0+cpu
 - typing-extensions==4.15.0


In [5]:
!uv add torch

Resolved 98 packages in 318ms                                        
Prepared 29 packages in 48.36s                                           
Installed 29 packages in 476ms                              
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.29.7
 + nvidia-nvjitlink==13.0.88
 + nvidia-nvshmem-cu13==3.4.5
 + nvidia-nvtx==13.0.85
 + setuptools==81.0.0
 + sympy==1.14.0
 + torch==2.12.0
 + triton==3.7.0
 + typing-extensions==4.15.0


In [6]:
# Cell 3: Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [7]:
!git pull

Already up to date.


In [8]:
!ls

cli.py	main.py    papers	   README.md  status_handoff.md
data	notebooks  pyproject.toml  src	      uv.lock


In [9]:
# Cell 4: Monotonicity ground truth audit (produces paper Section 4.1 data)
# Quick mode: --max-groups 5000 for a fast sample, remove flag for full audit
!uv run python cli.py analysis monotonicity --max-groups 5000

Loading data/processed/master.parquet...
Rows with numeric param_3 and valid targets: 33746
Total groups: 11703

Monotonicity Audit Summary (4920 consecutive pairs)
  Both hold (area↑ AND latency↓):   2539 (51.6%)
  Area only (area↑, latency↗):         8 (0.2%)
  Latency only (area↓, latency↓):   2363 (48.0%)
  Neither:                             10 (0.2%)

Conclusion: joint monotonicity holds in 51.6% of pairs.
The HINN penalty acts as a SOFT REGULARIZER (Bayesian prior), not a hard constraint.
Include this analysis in Section 4.1 of the manuscript.

Per-group audit saved to results/data/monotonicity_audit.csv


In [10]:
# Cell 5: Train HINN (primary model, seed=42)
# For multi-seed reporting: re-run with --seed 0, 7, 123, 999
!uv run python cli.py train train --epochs 300 --batch-size 1024 --seed 23

Seed: 23  |  Epochs: 300  |  Batch: 1024
Loading data...
Splits — Train: 35192, Val: 4400, Test: 4400
Scalers saved.
Device: cuda

Epoch  | LR       | Lambda | Train MSE  | Val MSE    | Val R2   | CVR (%) 
---------------------------------------------------------------------------
1      | 0.00100  | 0.00   | 0.6721     | 0.6133     | 0.4273   | 34.07   
10     | 0.00100  | 0.00   | 0.4946     | 0.5127     | 0.5193   | 36.48   
20     | 0.00099  | 0.00   | 0.3322     | 0.3119     | 0.7023   | 36.86   
30     | 0.00098  | 0.00   | 0.2515     | 0.2391     | 0.7684   | 35.09   
40     | 0.00096  | 0.04   | 0.2140     | 0.2190     | 0.7872   | 37.81   
50     | 0.00093  | 0.09   | 0.1986     | 0.2068     | 0.7986   | 37.13   
60     | 0.00091  | 0.13   | 0.1767     | 0.1999     | 0.8054   | 41.47   
70     | 0.00087  | 0.17   | 0.1633     | 0.1920     | 0.8128   | 40.01   
80     | 0.00084  | 0.21   | 0.1476     | 0.1892     | 0.8155   | 37.10   
90     | 0.00080  | 0.26   | 0.1324     | 0

In [11]:
# Cell 6: Train baselines (XGBoost + Vanilla MLP) — same data split
!uv run python cli.py train baselines --epochs 300 --seed 42


[XGBoost] seed=42, n_estimators=500, max_depth=7
  Training 4 XGBoost models (one per target) with early stopping...
  Test R² (overall): 0.6357
    hls_lut: 0.5497
    hls_ff: 0.6277
    average_latency: 0.7728
    best_latency: 0.5928

[Vanilla MLP] seed=42, epochs=300
  Epoch    1 | val_mse=0.5898
  Epoch   20 | val_mse=0.3101
  Epoch   40 | val_mse=0.2234
  Epoch   60 | val_mse=0.1987
  Epoch   80 | val_mse=0.1831
  Epoch  100 | val_mse=0.1691
  Epoch  120 | val_mse=0.1652
  Epoch  140 | val_mse=0.1564
  Epoch  160 | val_mse=0.1532
  Epoch  180 | val_mse=0.1458
  Epoch  200 | val_mse=0.1443
  Epoch  220 | val_mse=0.1458
  Epoch  240 | val_mse=0.1423
  Epoch  260 | val_mse=0.1395
  Epoch  280 | val_mse=0.1422
  Epoch  300 | val_mse=0.1392
  Test R² (overall): 0.8444
    hls_lut: 0.6781
    hls_ff: 0.7253
    average_latency: 0.9868
    best_latency: 0.9876

Baseline results saved to results/models/baseline_results.csv


In [12]:
# Cell 7: Run MOO — extract Pareto front from trained HINN surrogate
!uv run python cli.py moo run

Device: cuda
Loaded surrogate from results/models/hinn_best.pt
  Transform config: {'area_log1p_applied': True, 'lat_log1p_applied': True}
/content/2601_chip_paper/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Running surrogate inference over 43992 configurations...
Surrogate predictions saved to results/data/surrogate_predictions.csv
Surrogate Pareto front (9 points) saved to results/data/pareto_front.csv
Ground truth Pareto front (7 points) saved to results/data/gt_pareto_front.csv

Surrogate Pareto front range:
  Area:    [533, 153073] LUT+FF
  Latency: [18, 5297] cycles


In [26]:
# Cell 8: Generate all publication figures
!uv run python cli.py plot training-dynamics
!uv run python cli.py plot pareto
!uv run python cli.py plot monotonicity
!uv run python cli.py plot comparison

Saved training dynamics plot to results/figs/training_dynamics
Saved Pareto front plot to results/figs/pareto_front
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/2601_chip_paper/cli.py:148 in plot_monotonicity_cmd                 │
│                                                                              │
│   145 │   out: str = typer.Option("results/figs/monotonicity_proof",         │
│       "--out"),                                                              │
│   146 ) -> None:                                                             │
│   147 │   """Sweep parallelism factors through trained HINN to prove soft    │
│       monotonicity prior."""                                                 │
│ ❱ 148 │   from visualize.monotonicity_proof import plot_monotonicity         │
│   149 │   plot_monotonicity(model, "results/models/", features, out)         │
│   150                                                                   

In [13]:
# Cell 9: Zip all results (weights, scalers, CSVs, SVGs) and download
import shutil
import sys

# Include results/ subdirs: models/, figs/, data/
shutil.make_archive("hinn_results", "zip", "results")
print("Created hinn_results.zip in the current directory.")

if "google.colab" in sys.modules:
    try:
        from google.colab import files
        files.download("hinn_results.zip")
    except Exception as e:
        print(f"Could not auto-download (are you in VS Code?): {e}")
        print("Please download the zip manually from the file explorer.")
else:
    print("Not running in Colab web interface, skipping auto-download.")
    print("Please download the zip manually from the file explorer.")


Created hinn_results.zip in the current directory.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!git commit -m "results saved in colab"

[master 0932265] results saved in colab
 14 files changed, 49230 insertions(+)
 create mode 100644 results/data/gt_pareto_front.csv
 create mode 100644 results/data/monotonicity_audit.csv
 create mode 100644 results/data/pareto_front.csv
 create mode 100644 results/data/surrogate_predictions.csv
 create mode 100644 results/models/baseline_results.csv
 create mode 100644 results/models/hinn_best.pt
 create mode 100644 results/models/metrics.csv
 create mode 100644 results/models/scaler_X.pkl
 create mode 100644 results/models/scaler_y_area.pkl
 create mode 100644 results/models/scaler_y_lat.pkl
 create mode 100644 results/models/test_results.csv
 create mode 100644 results/models/transform_config.pkl
 create mode 100644 results/models/vanilla_mlp_best.pt
 create mode 100644 results/models/xgb_models.pkl


In [18]:
!git config --global user.email "thesattary@gmail.com"

!git config --global user.name "sat"



In [20]:
!git push origin master

fatal: could not read Username for 'https://github.com': No such device or address
